In [ ]:
"""
CELL 1: Configuration & Setup
Initializes libraries and global config for the scraper pipeline.
"""

import feedparser
import newspaper
import csv
import hashlib
import time
import requests
from datetime import datetime, timedelta

# News sources configuration
SOURCES_CONFIG = [
    {"domain_name": "Blognone", "base_url": "https://www.blognone.com"}, 
    {"domain_name": "TechCrunch", "base_url": "https://techcrunch.com"}, 
    {"domain_name": "The Verge", "base_url": "https://www.theverge.com"},
    {"domain_name": "BBC Tech", "base_url": "https://www.bbc.com"},
    {"domain_name": "Wired", "base_url": "https://www.wired.com"},
    {"domain_name": "Ars Technica", "base_url": "https://arstechnica.com"},
    {"domain_name": "Engadget", "base_url": "https://www.engadget.com"},
    {"domain_name": "The Information", "base_url": "https://www.theinformation.com"},
    {"domain_name": "CNBC Tech", "base_url": "https://www.cnbc.com/technology/"},
    {"domain_name": "VentureBeat", "base_url": "https://venturebeat.com"},
    {"domain_name": "Axios", "base_url": "https://www.axios.com"},
    {"domain_name": "Protocol", "base_url": "https://www.protocol.com"},
    {"domain_name": "The Markup", "base_url": "https://themarkup.org"},
    {"domain_name": "MIT Technology Review", "base_url": "https://www.technologyreview.com"},
]

# Keywords for article filtering
KEYWORDS_CONFIG = [
    "AI", "Artificial Intelligence", "Machine Learning", 
    "Data", "Google", "Microsoft", "Meta", "NVIDIA", "Crypto",
]

# Settings
DAYS_LOOKBACK = 14 
MIN_CONTENT_LENGTH = 300
OUTPUT_FILENAME = "scraped_data_from_homepages.csv"
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

print(f"\n{'='*70}")
print(f"✓ Config loaded: {len(SOURCES_CONFIG)} sources, {len(KEYWORDS_CONFIG)} keywords")
print(f"  Lookback: {DAYS_LOOKBACK} days | Min content: {MIN_CONTENT_LENGTH} chars")
print(f"{'='*70}\n")


Configuration Loaded. Targets: ['Blognone', 'TechCrunch', 'The Verge', 'BBC Tech', 'Wired', 'Ars Technica', 'Engadget', 'The Information', 'CNBC Tech', 'VentureBeat', 'Axios', 'Protocol', 'The Markup', 'MIT Technology Review']


In [ ]:
"""
CELL 2: RSS Feed Discovery
Discovers RSS feed URLs from website homepages by testing common locations.
"""

def resolve_rss_link(homepage_url):
    """
    Discover RSS feed URL from homepage.
    
    Tries 7 common RSS locations; validates with feedparser.
    Returns URL if found, None otherwise.
    """
    print(f"   🔎 Resolving: {homepage_url}...", end="")
    
    possible_suffixes = [
        "", "/feed", "/rss", "/rss.xml", "/atom.xml", "/feed.xml", "/index.xml"
    ]
    
    headers = {'User-Agent': USER_AGENT}

    for suffix in possible_suffixes:
        target = homepage_url.rstrip("/") + suffix
        try:
            response = requests.get(target, headers=headers, timeout=3)
            if response.status_code == 200:
                d = feedparser.parse(response.content)
                if len(d.entries) > 0:
                    print(f" ✓ FOUND")
                    return target
        except Exception:
            continue
            
    print(" ✗")
    return None


In [ ]:
"""
CELL 3: Helper Functions
Utilities for URL hashing, date validation, and keyword matching.
"""

def generate_hash(url):
    """SHA-256 hash for URL deduplication."""
    return hashlib.sha256(url.encode('utf-8')).hexdigest()

def is_date_valid(published_struct):
    """Check if article is within DAYS_LOOKBACK window."""
    if not published_struct: 
        return True
    pub_date = datetime(*published_struct[:6])
    cutoff = datetime.now() - timedelta(days=DAYS_LOOKBACK)
    return pub_date >= cutoff

def get_matched_keywords(headline, text, keyword_list):
    """Find keywords in article text (case-insensitive)."""
    combined = (headline + " " + text).lower()
    matches = [kw for kw in keyword_list if kw.lower() in combined]
    return matches if matches else []

def normalize_source_name(source_name):
    """Normalize source name for database lookup."""
    return source_name.strip()


In [ ]:
"""
CELL 4: RSS Pipeline
Main scraper: discovers feeds, extracts articles, filters by date/length/keywords.
Outputs CSV with database-ready fields (url_hash, full_content, matched_tags, status).
"""

def run_test_pipeline():
    """Execute RSS scraper across all sources."""
    all_data = []
    
    for source in SOURCES_CONFIG:
        print(f"\n--- {source['domain_name']} ---")
        
        # Discover RSS feed
        feed_url = resolve_rss_link(source['base_url'])
        if not feed_url:
            print("   [Skip] No RSS feed found")
            continue
            
        # Parse feed
        feed = feedparser.parse(feed_url)
        print(f"   📡 {len(feed.entries)} entries")
        
        for entry in feed.entries:
            try:
                headline = entry.title
                article_url = entry.link
                
                # Filter 1: Date range
                if hasattr(entry, 'published_parsed') and not is_date_valid(entry.published_parsed):
                    continue

                # Extract content
                art = newspaper.Article(article_url)
                art.download()
                art.parse()
                
                # Filter 2: Content length
                if not art.text or len(art.text) < MIN_CONTENT_LENGTH:
                    continue
                    
                # Filter 3: Keywords
                matched_keywords = get_matched_keywords(headline, art.text, KEYWORDS_CONFIG)
                if not matched_keywords:
                    continue
                
                # Prepare database-ready row
                url_hash = generate_hash(article_url)
                published_str = entry.published if hasattr(entry, 'published') else datetime.now().isoformat()
                
                row = {
                    "source": source['domain_name'],
                    "headline": headline,
                    "author": art.authors[0] if art.authors else "Unknown",
                    "url": article_url,
                    "published": published_str,
                    "keywords": ", ".join(matched_keywords),
                    "content_snippet": art.text[:100].replace('\n', ' ') + "...",
                    "url_hash": url_hash,
                    "full_content": art.text,
                    "matched_tags": matched_keywords,
                    "status": "New",
                }
                all_data.append(row)
                print(f"   ✓ {headline[:50]}...")
                
            except Exception:
                continue

    # Export to CSV
    if all_data:
        keys = all_data[0].keys()
        with open(OUTPUT_FILENAME, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(all_data)
        
        print(f"\n{'='*70}")
        print(f"✨ RSS COMPLETE: {len(all_data)} articles → {OUTPUT_FILENAME}")
        print(f"   Fields: url_hash, full_content, matched_tags, status")
        print(f"{'='*70}\n")
    else:
        print("\n⚠️ No articles found")

run_test_pipeline()



--- Processing Source: Blognone ---
   🔎 Resolving Feed for: https://www.blognone.com... FOUND! -> https://www.blognone.com/atom.xml
   📡 Scanning 10 entries...
 FOUND! -> https://www.blognone.com/atom.xml
   📡 Scanning 10 entries...

--- Processing Source: TechCrunch ---
   🔎 Resolving Feed for: https://techcrunch.com...
--- Processing Source: TechCrunch ---
   🔎 Resolving Feed for: https://techcrunch.com... FOUND! -> https://techcrunch.com/feed
 FOUND! -> https://techcrunch.com/feed
   📡 Scanning 20 entries...
   📡 Scanning 20 entries...
   ✅ Keep: Claude Code is coming to Slack, and that...
   ✅ Keep: Claude Code is coming to Slack, and that...
   ✅ Keep: Google details security measures for Chr...
   ✅ Keep: Google details security measures for Chr...
   ✅ Keep: You can buy your Instacart groceries wit...
   ✅ Keep: You can buy your Instacart groceries wit...
   ✅ Keep: TikTok adds a space for organizing conte...
   ✅ Keep: TikTok adds a space for organizing conte...
   ✅ Keep: Pe

# CELL 5: HTML Fallback Scraper

Discovers articles from sources without RSS feeds:
1. **Robots.txt → Sitemap** (primary strategy)
2. **Seed pages** (home + /news + /tech) + regex link extraction (fallback)
3. Extract content with newspaper3k; filter by length and keywords

**Constraints**: Same-domain only, max 5 seed pages, 20 articles/source, 3s timeout.

**Output**: `scraped_data_html_fallback.csv` (same schema as RSS)


In [ ]:
"""
CELL 6: HTML Fallback Pipeline
Scrapes articles from sources without RSS feeds.
Tries sitemaps (robots.txt) first; falls back to seed page crawling.
"""

import re
import xml.etree.ElementTree as ET
from urllib.parse import urljoin, urlparse

HTML_OUTPUT_FILENAME = "scraped_data_html_fallback.csv"
MAX_SEED_PAGES = 5
MAX_ARTICLES_PER_SOURCE = 20
REQUEST_TIMEOUT = 3

def fetch_url(url, headers=None):
    """Safe HTTP GET with rate-limit awareness (respects 429/503)."""
    try:
        resp = requests.get(url, headers=headers or {'User-Agent': USER_AGENT}, timeout=REQUEST_TIMEOUT)
        if resp.status_code in (429, 503):
            return None
        if resp.status_code != 200:
            return None
        return resp.text
    except Exception:
        return None

def discover_sitemaps(base_url):
    """Extract sitemap URLs from robots.txt."""
    robots_url = urljoin(base_url, "/robots.txt")
    text = fetch_url(robots_url)
    if not text:
        return []
    sitemaps = []
    for line in text.splitlines():
        if line.lower().startswith("sitemap:"):
            parts = line.split(":", 1)
            if len(parts) == 2:
                sm_url = parts[1].strip()
                if sm_url:
                    sitemaps.append(sm_url)
    return sitemaps

def parse_sitemap_urls(sitemap_url, limit=30):
    """Extract URLs from XML sitemap."""
    xml_text = fetch_url(sitemap_url)
    if not xml_text:
        return []
    try:
        root = ET.fromstring(xml_text)
        ns = {'sm': root.tag.split('}')[0].strip('{') if '}' in root.tag else ''}
        urls = []
        for url in root.findall('sm:url', ns):
            loc = url.find('sm:loc', ns)
            if loc is not None and loc.text:
                urls.append(loc.text.strip())
            if len(urls) >= limit:
                break
        return urls
    except Exception:
        return []

def extract_links_from_html(base_url, html_text, limit=50):
    """Extract article-like links from HTML using regex heuristics.
    
    Filters: same-domain, article patterns (/news/, /article/, date formats),
    excludes login/subscribe/tags/media.
    """
    links = set()
    if not html_text:
        return []
    for match in re.finditer(r'href=["\'](.*?)["\']', html_text, re.IGNORECASE):
        href = match.group(1)
        if href.startswith('#'):
            continue
        abs_url = urljoin(base_url, href)
        parsed = urlparse(abs_url)
        if parsed.netloc != urlparse(base_url).netloc:
            continue
        if any(seg in abs_url for seg in ["/login", "/subscribe", "/tag/", "/category/", "/search", ".pdf", ".mp4", ".jpg", ".png"]):
            continue
        if not re.search(r"(\d{4}/\d{2}/\d{2})|(/news/)|(/article/)|(/story/)", abs_url):
            continue
        links.add(abs_url)
        if len(links) >= limit:
            break
    return list(links)

def get_seed_pages(base_url):
    """Return homepage variants to crawl."""
    return [
        base_url,
        urljoin(base_url, "/news"),
        urljoin(base_url, "/tech"),
        urljoin(base_url, "/technology"),
    ][:MAX_SEED_PAGES]

def run_html_fallback_test():
    """Execute HTML fallback scraper."""
    all_rows = []
    for source in SOURCES_CONFIG:
        domain = source["domain_name"]
        base = source["base_url"]
        print(f"\n[HTML] {domain}")
        collected = []

        # Strategy 1: Sitemaps
        sitemaps = discover_sitemaps(base)
        for sm in sitemaps:
            urls = parse_sitemap_urls(sm, limit=MAX_ARTICLES_PER_SOURCE)
            collected.extend(urls)
            if len(collected) >= MAX_ARTICLES_PER_SOURCE:
                break

        # Strategy 2: Seed pages
        if len(collected) < MAX_ARTICLES_PER_SOURCE:
            for seed in get_seed_pages(base):
                html = fetch_url(seed)
                links = extract_links_from_html(base, html, limit=MAX_ARTICLES_PER_SOURCE - len(collected))
                collected.extend(links)
                if len(collected) >= MAX_ARTICLES_PER_SOURCE:
                    break

        collected = list(dict.fromkeys(collected))
        print(f"   Links: {len(collected)}")

        # Extract articles
        for url in collected[:MAX_ARTICLES_PER_SOURCE]:
            try:
                art = newspaper.Article(url)
                art.download()
                art.parse()
                if not art.text or len(art.text) < MIN_CONTENT_LENGTH:
                    continue
                matched_tags = get_matched_keywords(art.title or "", art.text, KEYWORDS_CONFIG)
                if not matched_tags:
                    continue
                row = {
                    "source": domain,
                    "headline": art.title or "(no title)",
                    "author": art.authors[0] if art.authors else "Unknown",
                    "url": url,
                    "published": art.publish_date or datetime.now(),
                    "keywords": ", ".join(matched_tags),
                    "content_snippet": art.text[:100].replace('\n', ' ') + "...",
                    "url_hash": generate_hash(url),
                    "full_content": art.text,
                    "matched_tags": matched_tags,
                    "status": "New"
                }
                all_rows.append(row)
                print(f"   ✓ {row['headline'][:50]}...")
            except Exception:
                continue

    # Export to CSV
    if all_rows:
        keys = all_rows[0].keys()
        with open(HTML_OUTPUT_FILENAME, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(all_rows)
        print(f"\n{'='*70}")
        print(f"✨ HTML COMPLETE: {len(all_rows)} articles → {HTML_OUTPUT_FILENAME}")
        print(f"{'='*70}\n")
    else:
        print("\n⚠️ No articles found")

run_html_fallback_test()



[HTML] Source: Blognone
   Links collected: 0

[HTML] Source: TechCrunch
   Links collected: 0

[HTML] Source: TechCrunch
   Links collected: 19
   Links collected: 19
   ✅ Claude Code is coming to Slack, and that’s a bigger deal tha...
   ✅ Claude Code is coming to Slack, and that’s a bigger deal tha...
   ✅ Google details security measures for Chrome’s agentic featur...
   ✅ Google details security measures for Chrome’s agentic featur...
   ✅ You can buy your Instacart groceries without leaving ChatGPT...
   ✅ You can buy your Instacart groceries without leaving ChatGPT...
   ✅ TikTok adds a space for organizing content with others, teas...
   ✅ TikTok adds a space for organizing content with others, teas...
   ✅ Petco’s security lapse affected customers’ SSNs, driver’s li...
   ✅ Petco’s security lapse affected customers’ SSNs, driver’s li...
   ✅ Heat pump startup Quilt raises $20M Series B to expand sales...
   ✅ Heat pump startup Quilt raises $20M Series B to expand sales...
   